# ARCH-04 — distribuição de duração de áudio do dataset EXIST 2026 Task 3

**Por que este notebook existe.** O finding ARCH-04
(`improvements/01_ARCHITECTURE_FINDINGS.md`) afirma que as features de áudio do
Whisper estão *diluídas por padding* e *truncadas em 30 s*. As percentagens que
sustentam essa afirmação (mediana 16,4 s; 61,7 % < 30 s; 45 % < 15 s; fração
válida média ≈ 0,63) vieram de uma **amostra de 60 vídeos de treino**, não do
dataset inteiro.

Este notebook **mede a população completa** e ou confirma ou substitui essa base
de evidência.

## O que exatamente é medido

`whisper.pad_or_trim` opera sobre a forma de onda que `torchaudio.load` devolve —
ou seja, sobre o **stream de áudio**, não sobre o container mp4. Os dois quase
sempre coincidem, mas podem divergir, e a afirmação inteira é "que fração da
janela de 30 s é áudio real". Portanto medimos o stream:

```
ffprobe -v error -select_streams a:0 -show_entries stream=duration -of csv=p=0 <file>
```

**Consequência que um script ingénuo perde:** arquivos **sem stream de áudio**
devolvem saída vazia. Esses são 100 % padding — o pior caso da distribuição
inteira — e desapareceriam das percentagens se virassem `NaN`. São contados
como categoria própria.

## Quantidades derivadas

Para cada item, com `d` = duração do áudio em segundos e janela `W = 30 s`:

$$
\text{valid\_fraction} = \frac{\min(d,\ W)}{W}
\qquad
\text{discarded\_fraction} = \frac{\max(d - W,\ 0)}{d}
$$

- `valid_fraction` — quanto da janela de 30 s promediada pelo encoder é áudio
  real. É o número 0,63 citado no ARCH-04. `1 - valid_fraction` é a fração do
  vetor `[1280]` que é encoder de silêncio.
- `discarded_fraction` — quanto do áudio real nunca chega ao encoder, por corte.

Os dois defeitos são mutuamente exclusivos por item (`d < 30` dilui, `d > 30`
trunca) mas coexistem no dataset.

In [ ]:
import json
import subprocess
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Raiz do EXIST_VIDEOS (este notebook vive em EXIST_VIDEOS/analysis/)
ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path("EXIST_VIDEOS")
assert (ROOT / "Dataset").exists(), f"Dataset nao encontrado a partir de {ROOT}"

WHISPER_WINDOW_S = 30.0   # whisper.pad_or_trim: 480000 amostras @ 16 kHz
TARGET_SR = 16000

# Mesma resolucao de caminho que extract_audio.ipynb usa:
#   video_path = VIDEO_DIR / Path(row["path_video"]).name
SPLITS = {
    "training": (ROOT / "Dataset/training/EXIST2026_training.json",
                 ROOT / "Dataset/training/videos"),
    "test":     (ROOT / "Dataset/test/EXIST2026_test_clean.json",
                 ROOT / "Dataset/test/videos"),
}

OUT_CSV = ROOT / "analysis/audio_durations.csv"
print(f"ROOT = {ROOT.resolve()}")

## 1. Iterar pelos manifestos, não pelo glob de disco

A junção é feita por `id_EXIST` → `Path(row["path_video"]).name`, exatamente como
`extract_audio.ipynb` faz. Isto é o que **expõe** divergências entre manifesto e
disco, em vez de escondê-las: um `glob` de `*.mp4` contaria arquivos que nenhuma
linha do manifesto referencia, e não notaria linhas cujo vídeo não existe.

In [ ]:
def probe_audio_duration(path: Path):
    '''Duracao do stream de audio em segundos.

    Devolve (status, duracao):
      ("ok", float)        stream de audio presente e legivel
      ("no_audio", nan)    arquivo existe mas nao tem stream de audio -> 100% padding
      ("probe_failed", nan) ffprobe falhou (arquivo corrompido)
    '''
    try:
        out = subprocess.run(
            ["ffprobe", "-v", "error", "-select_streams", "a:0",
             "-show_entries", "stream=duration", "-of", "csv=p=0", str(path)],
            capture_output=True, text=True, timeout=60,
        )
    except subprocess.TimeoutExpired:
        return ("probe_failed", float("nan"))
    raw = out.stdout.strip()
    if out.returncode != 0:
        return ("probe_failed", float("nan"))
    if not raw or raw.upper() == "N/A":
        # Sem stream de audio, OU stream sem duracao declarada no header.
        # Fallback: perguntar ao container so pelo stream de audio.
        alt = subprocess.run(
            ["ffprobe", "-v", "error", "-select_streams", "a:0",
             "-show_entries", "format=duration", "-of", "csv=p=0", str(path)],
            capture_output=True, text=True, timeout=60,
        )
        has_audio = subprocess.run(
            ["ffprobe", "-v", "error", "-select_streams", "a:0",
             "-show_entries", "stream=index", "-of", "csv=p=0", str(path)],
            capture_output=True, text=True, timeout=60,
        ).stdout.strip()
        if not has_audio:
            return ("no_audio", float("nan"))
        try:
            return ("ok_container_fallback", float(alt.stdout.strip()))
        except ValueError:
            return ("probe_failed", float("nan"))
    try:
        return ("ok", float(raw))
    except ValueError:
        return ("probe_failed", float("nan"))


rows = []
for split, (json_path, video_dir) in SPLITS.items():
    if not json_path.exists():
        print(f"[SKIP] {split}: manifesto ausente ({json_path})")
        continue
    df_manifest = pd.read_json(json_path).transpose()
    print(f"[{split}] {len(df_manifest)} linhas no manifesto")

    tasks = []
    for _, row in df_manifest.iterrows():
        item_id = str(row["id_EXIST"])
        vpath = video_dir / Path(str(row["path_video"])).name
        tasks.append((split, item_id, vpath))

    def work(t):
        split_, item_id, vpath = t
        if not vpath.exists():
            return {"split": split_, "id_EXIST": item_id, "file": vpath.name,
                    "status": "file_missing", "duration_s": float("nan")}
        status, dur = probe_audio_duration(vpath)
        return {"split": split_, "id_EXIST": item_id, "file": vpath.name,
                "status": status, "duration_s": dur}

    with ThreadPoolExecutor(max_workers=8) as ex:
        rows.extend(ex.map(work, tasks))

df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)
print(f"\nGravado: {OUT_CSV}  ({len(df)} linhas)")

## 2. Reconciliação de contagens — antes de qualquer percentagem

Uma percentagem calculada sobre um denominador errado é pior que nenhuma
percentagem. Esta tabela tem de fechar antes de olhar para qualquer histograma.

In [ ]:
recon = (df.groupby(["split", "status"]).size()
           .unstack(fill_value=0))
recon["TOTAL_manifesto"] = recon.sum(axis=1)
print("Reconciliacao por split e status:\n")
print(recon.to_string())

# Contraste com o que existe em disco, fora do manifesto
n_mp4_disk = len(list((ROOT / "Dataset").rglob("*.mp4")))
n_pt_audio = len(list((ROOT / "audio_features").rglob("*.pt"))) \
    if (ROOT / "audio_features").exists() else 0
print(f"\nmp4 em disco (rglob):        {n_mp4_disk}")
print(f"linhas de manifesto:         {len(df)}")
print(f"features .pt de audio:       {n_pt_audio}")
if n_mp4_disk != len(df):
    print(f"  -> DIVERGENCIA de {n_mp4_disk - len(df)} arquivo(s): "
          "mp4 em disco que nenhuma linha do manifesto referencia, "
          "ou vice-versa. Investigar antes de citar percentagens.")

## 3. Estatísticas por split

Treino e teste são populações diferentes. A afirmação do ARCH-04 é sobre as
features de **treino** — juntar os dois num histograma só responderia a uma
pergunta que ninguém fez. Ambos são reportados, separados.

O `no_audio` entra na contagem de "afetados por padding" com `valid_fraction = 0`
(é o pior caso possível), mas fica **fora** das estatísticas de duração, porque
não tem duração.

In [ ]:
def summarize(sub: pd.DataFrame, label: str):
    ok = sub[sub["status"].str.startswith("ok")].copy()
    n_total = len(sub)
    n_ok = len(ok)
    n_noaudio = int((sub["status"] == "no_audio").sum())
    n_missing = int((sub["status"] == "file_missing").sum())
    n_failed = int((sub["status"] == "probe_failed").sum())

    d = ok["duration_s"].to_numpy()
    ok["valid_fraction"] = np.minimum(d, WHISPER_WINDOW_S) / WHISPER_WINDOW_S
    ok["discarded_fraction"] = np.maximum(d - WHISPER_WINDOW_S, 0.0) / d

    print(f"===== {label} =====")
    print(f"n manifesto            {n_total}")
    print(f"n mensuravel           {n_ok}")
    print(f"n sem stream de audio  {n_noaudio}   <- 100% padding")
    print(f"n arquivo ausente      {n_missing}")
    print(f"n ffprobe falhou       {n_failed}")
    if n_ok == 0:
        return ok
    print()
    for q, name in [(0.10, "p10"), (0.25, "p25"), (0.50, "mediana"),
                    (0.75, "p75"), (0.90, "p90")]:
        print(f"  {name:8s} {np.quantile(d, q):8.2f} s")
    print(f"  {'media':8s} {d.mean():8.2f} s")
    print(f"  {'min':8s} {d.min():8.2f} s")
    print(f"  {'max':8s} {d.max():8.2f} s")
    print()
    print(f"  < 30 s (diluidos)    {(d < 30).sum():5d}  "
          f"({100 * (d < 30).mean():5.1f} %)")
    print(f"  < 15 s               {(d < 15).sum():5d}  "
          f"({100 * (d < 15).mean():5.1f} %)")
    print(f"  > 30 s (truncados)   {(d > 30).sum():5d}  "
          f"({100 * (d > 30).mean():5.1f} %)")
    print()
    print(f"  valid_fraction media {ok['valid_fraction'].mean():.4f}")
    print(f"  -> {100 * (1 - ok['valid_fraction'].mean()):.1f} % do vetor [1280] "
          "medio e encoder de silencio")
    trunc = ok[ok["discarded_fraction"] > 0]
    if len(trunc):
        print(f"  entre os truncados, audio descartado: "
              f"media {100 * trunc['discarded_fraction'].mean():.1f} %, "
              f"max {100 * trunc['discarded_fraction'].max():.1f} %")
    print()
    return ok


per_split = {}
for split in df["split"].unique():
    per_split[split] = summarize(df[df["split"] == split], split.upper())

## 4. Comparação com a amostra n=60 do ARCH-04

O objetivo desta tabela é uma coisa só: decidir se a evidência citada no
finding continua de pé. Se os números da população completa ficarem próximos dos
da amostra, a amostra segurou e o finding **fica mais firme**, não mais fraco.

In [ ]:
ARCH04_SAMPLE = {   # amostra n=60, treino, ffprobe (01_ARCHITECTURE_FINDINGS.md)
    "mediana (s)":        16.4,
    "media (s)":          25.3,
    "p10 (s)":             7.8,
    "max (s)":            59.1,
    "% < 30 s":           61.7,
    "% < 15 s":           45.0,
    "% > 30 s":           38.0,
    "valid_fraction":      0.63,
}

tr = per_split.get("training")
if tr is not None and len(tr):
    d = tr["duration_s"].to_numpy()
    full = {
        "mediana (s)":     float(np.median(d)),
        "media (s)":       float(d.mean()),
        "p10 (s)":         float(np.quantile(d, 0.10)),
        "max (s)":         float(d.max()),
        "% < 30 s":        float(100 * (d < 30).mean()),
        "% < 15 s":        float(100 * (d < 15).mean()),
        "% > 30 s":        float(100 * (d > 30).mean()),
        "valid_fraction":  float(tr["valid_fraction"].mean()),
    }
    cmp_df = pd.DataFrame({
        "ARCH-04 (n=60)": ARCH04_SAMPLE,
        f"populacao (n={len(d)})": full,
    })
    cmp_df["delta"] = cmp_df.iloc[:, 1] - cmp_df.iloc[:, 0]
    cmp_df["delta_rel_%"] = 100 * cmp_df["delta"] / cmp_df.iloc[:, 0]
    print(cmp_df.round(3).to_string())

## 5. Gráficos

In [ ]:
splits = [s for s in ["training", "test"] if s in per_split and len(per_split[s])]
fig, axes = plt.subplots(len(splits), 3, figsize=(16, 4.2 * len(splits)),
                         squeeze=False)

for r, split in enumerate(splits):
    ok = per_split[split]
    d = ok["duration_s"].to_numpy()

    # --- (a) histograma de duracao -------------------------------------
    ax = axes[r][0]
    ax.hist(d, bins=60, color="#4C72B0", edgecolor="white", linewidth=0.4)
    ax.axvline(WHISPER_WINDOW_S, color="#C44E52", lw=2)
    ax.annotate("janela do Whisper = 30 s", xy=(WHISPER_WINDOW_S, ax.get_ylim()[1] * 0.92),
                xytext=(6, 0), textcoords="offset points",
                color="#C44E52", fontsize=9, va="top")
    ax.axvline(np.median(d), color="#55A868", lw=1.6, ls="--")
    ax.annotate(f"mediana = {np.median(d):.1f} s",
                xy=(np.median(d), ax.get_ylim()[1] * 0.72),
                xytext=(-6, 0), textcoords="offset points",
                color="#55A868", fontsize=9, ha="right", va="top")
    ax.set_title(f"{split} — duração do stream de áudio (n={len(d)})")
    ax.set_xlabel("segundos")
    ax.set_ylabel("nº de vídeos")

    # --- (b) as tres populacoes ----------------------------------------
    ax = axes[r][1]
    n_dil = int((d < 30).sum())
    n_exact = int(((d >= 30) & (d <= 30)).sum())
    n_trunc = int((d > 30).sum())
    n_noaudio = int((df[(df["split"] == split)]["status"] == "no_audio").sum())
    labels = ["sem áudio\n(100 % padding)", "< 30 s\n(diluído)", "> 30 s\n(truncado)"]
    vals = [n_noaudio, n_dil, n_trunc]
    colors = ["#8172B2", "#DD8452", "#C44E52"]
    bars = ax.bar(labels, vals, color=colors)
    denom = len(d) + n_noaudio
    for b, v in zip(bars, vals):
        ax.annotate(f"{v}\n({100 * v / denom:.1f} %)",
                    xy=(b.get_x() + b.get_width() / 2, v),
                    xytext=(0, 4), textcoords="offset points",
                    ha="center", fontsize=9)
    ax.set_title(f"{split} — quantos itens cada defeito atinge")
    ax.set_ylabel("nº de vídeos")
    ax.set_ylim(0, max(vals) * 1.22 if max(vals) else 1)

    # --- (c) valid_fraction --------------------------------------------
    ax = axes[r][2]
    vf = ok["valid_fraction"].to_numpy()
    ax.hist(vf, bins=40, color="#937860", edgecolor="white", linewidth=0.4)
    ax.axvline(vf.mean(), color="#C44E52", lw=2)
    ax.annotate(f"média = {vf.mean():.3f}\n→ {100 * (1 - vf.mean()):.0f} % do vetor\né silêncio",
                xy=(vf.mean(), ax.get_ylim()[1] * 0.9),
                xytext=(-8, 0), textcoords="offset points",
                color="#C44E52", fontsize=9, ha="right", va="top")
    ax.set_title(f"{split} — fração válida da janela de 30 s")
    ax.set_xlabel("min(duração, 30) / 30")
    ax.set_ylabel("nº de vídeos")

fig.suptitle("ARCH-04 — diluição por padding e truncamento nas features de áudio do Whisper",
             fontsize=13, y=1.005)
fig.tight_layout()
fig.savefig(ROOT / "analysis/arch04_audio_durations.png", dpi=150,
            bbox_inches="tight")
plt.show()

### 5.1 CDF — leitura direta de "quantos % abaixo de X segundos"

O histograma mostra a forma; a CDF responde à pergunta operacional. A linha
vertical em 30 s cruza a curva exatamente na percentagem de itens diluídos.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.2))
for split, color in zip(splits, ["#4C72B0", "#DD8452"]):
    d = np.sort(per_split[split]["duration_s"].to_numpy())
    y = np.arange(1, len(d) + 1) / len(d) * 100
    ax.plot(d, y, lw=2, color=color, label=f"{split} (n={len(d)})")
    frac30 = 100 * (d < 30).mean()
    ax.plot([30], [frac30], "o", color=color, ms=7)
    ax.annotate(f"{frac30:.1f} %", xy=(30, frac30), xytext=(8, -4),
                textcoords="offset points", color=color, fontsize=10)

ax.axvline(30, color="#C44E52", lw=1.8, ls="--")
ax.annotate("janela do Whisper", xy=(30, 4), xytext=(6, 0),
            textcoords="offset points", color="#C44E52", fontsize=9)
ax.axvline(15, color="#999999", lw=1.2, ls=":")
ax.annotate("15 s", xy=(15, 4), xytext=(4, 0), textcoords="offset points",
            color="#666666", fontsize=9)
ax.set_xlabel("duração do stream de áudio (s)")
ax.set_ylabel("% de vídeos com duração ≤ x")
ax.set_title("CDF da duração de áudio — a linha de 30 s corta a fração diluída")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(ROOT / "analysis/arch04_audio_cdf.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Onde ficam os artefatos

| arquivo | o que é |
|---|---|
| `analysis/audio_durations.csv` | uma linha por item do manifesto: `split`, `id_EXIST`, `file`, `status`, `duration_s` |
| `analysis/arch04_audio_durations.png` | painel de 3 colunas por split |
| `analysis/arch04_audio_cdf.png` | CDF |

O CSV fica em `analysis/`, **não** em `Dataset/` — `Dataset/` está no
`.gitignore` com uma exceção estreita só para `val_split_*.json` (Decisão 9 do
`05_IMPLEMENTATION_NOTES.md`), então um CSV ali seria invisível ao git.

## 7. O que estes números NÃO dizem

- **Não medem ganho de métrica.** Que a re-extração corrigida melhore o ICM
  continua a ser hipótese não verificada. Só um re-treino confirma.
- **Não validam o alinhamento áudio-vídeo.** Frames e áudio são amostrados por
  caminhos independentes; nada aqui testa se concordam temporalmente.
- **Duração do stream ≠ duração de fala.** Um vídeo de 40 s pode ter 5 s de fala
  e 35 s de música. `valid_fraction` mede quanto da janela é *sinal*, não quanto
  é *conteúdo linguístico*. O defeito medido é real e é um limite inferior do
  dano; a diluição efetiva pode ser pior.